# <span style="color:red">HW2_Reza_Zamani</span>

# <span style="color:green">This project is for Python for MFE program at UC Berkeley </span>

In [43]:
# ----------------------------
# Imports
# ----------------------------
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# Notebook / plotting config
# ----------------------------
%matplotlib inline

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
    "grid.alpha": 0.3,
})


# ----------------------------
# Reproducibility
# ----------------------------
RANDOM_SEED = 18
np.random.seed(RANDOM_SEED)


# Part 1
## Data Understanding 


In [44]:
# ----------------------------
# Connect to database
# ----------------------------
DB_PATH = "data3.db"
TABLE = "ohlc_hw"

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql(
        f"""
        SELECT
            ts,
            open,
            high,
            low,
            close,
            adj_close,
            volume,
            ticker
        FROM {TABLE}
        ORDER BY ts
        """,
        conn
    )

# ----------------------------
# Basic type handling
# ----------------------------
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

# ----------------------------
# Quick inspection
# ----------------------------
display(df.head())
df.info()
df.describe(include="all")


,ts,open,high,low,close,adj_close,volume,ticker
0,1962-01-02,7.370000,7.370000,7.290000,7.290000,1.540000,407940,IBM
1,1962-01-02,0.000000,1.440000,1.420000,1.430000,0.259300,192000,PG
2,1962-01-02,0.837449,0.837449,0.823045,0.823045,0.190931,352350,BA
3,1962-01-02,0.263020,0.270180,0.263020,0.263020,0.048145,806400,KO
4,1962-01-02,3.550000,3.550000,3.450000,3.470000,0.719220,254522,"<a target=""_blank""><span style=""color: red"">MM..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05,397711
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,150
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PG
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15450
mean,1998-04-17 21:18:45.292486144,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07,NaN
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06,NaN
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07,NaN
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09,NaN


In [45]:
# ----------------------------
# ----------------------------

#  dataset structure
# ----------------------------
# ----------------------------

print("Columns:")
for col in df.columns:
    print(f"  - {col}")


Columns:
  - ts
  - open
  - high
  - low
  - close
  - adj_close
  - volume
  - ticker


## Column descriptions
### here we want to check the columns 

- **ts**: date and time
- **open**: opening price
- **high**: highest price of the day
- **low**: lowest price of the day
- **close**: closing price
- **adj_close**: adjusted closing price
- **volume**: number of shares traded
- **ticker**: stock identifier


In [46]:

start_date = df["ts"].min()
end_date = df["ts"].max()

print(f"Date range: {start_date:%Y-%m-%d} to {end_date:%Y-%m-%d}")

# Clean ticker names (remove stray markup, if any)
df["ticker"] = df["ticker"].astype(str).str.replace(r"<.*?>", "", regex=True).str.strip()

tickers = sorted(df["ticker"].unique())
print(f"Tickers ({len(tickers)}): {tickers}")


Date range: 1962-01-02 to 2024-12-06
Tickers (30): ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


## Data Quality 
**Missing Data**

In [47]:
# ----------------------------
# Missing data & trading-day checks
# ----------------------------
def missing_data_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reports:
    1) Missing values per column
    2) Missing trading days per ticker (business days)
    """

    # Work on a safe copy
    data = df.copy()
    data["ts"] = pd.to_datetime(data["ts"], errors="coerce")

    # ----------------------------
    # Missing values per column
    # ----------------------------
    missing_by_col = data.isna().sum()
    print("Missing values per column:")
    display(missing_by_col[missing_by_col > 0])

    # ----------------------------
    # Missing trading days per ticker
    # ----------------------------
    records = []

    for ticker, g in data.dropna(subset=["ts"]).groupby("ticker"):
        g = g.sort_values("ts")

        start_date = g["ts"].min().normalize()
        end_date = g["ts"].max().normalize()

        expected_days = pd.bdate_range(start=start_date, end=end_date)
        actual_days = pd.DatetimeIndex(g["ts"].dt.normalize().unique())

        missing_days = expected_days.difference(actual_days)

        if not missing_days.empty:
            records.append({
                "ticker": ticker,
                "missing_days": len(missing_days),
                "start_date": start_date,
                "end_date": end_date,
                "total_expected": len(expected_days),
                "total_actual": len(actual_days),
            })

    result = pd.DataFrame(records).sort_values("ticker").reset_index(drop=True)

    # ----------------------------
    # Summary output
    # ----------------------------
    if result.empty:
        print("\n✓ No missing trading days detected.")
    else:
        print("\nMissing trading days summary:")
        display(result)

        for _, row in result.iterrows():
            print(
                f"Ticker {row['ticker']} is missing {row['missing_days']} trading days "
                f"between {row['start_date'].date()} and {row['end_date'].date()}."
            )

    return result


# Run report
missing_trading_days = missing_data_report(df)


Missing values per column:


adj_close    19855
dtype: int64


Missing trading days summary:


,ticker,missing_days,start_date,end_date,total_expected,total_actual
0,AAPL,391,1980-12-12,2024-10-18,11441,11050
1,AMGN,365,1983-06-17,2024-10-18,10786,10421
2,AMZN,255,1997-05-15,2024-12-06,7192,6937
3,AXP,482,1972-04-03,2024-10-18,13710,13252
4,BA,600,1962-01-02,2024-10-18,16384,15784
5,CAT,600,1962-01-02,2024-10-18,16384,15808
6,CRM,186,2004-06-23,2024-10-18,5303,5117
7,CSCO,312,1990-02-16,2024-10-18,9046,8734
8,CVX,600,1962-01-02,2024-12-06,16419,15843
9,DIS,576,1962-01-02,2024-12-06,16419,15867


Ticker AAPL is missing 391 trading days between 1980-12-12 and 2024-10-18.
Ticker AMGN is missing 365 trading days between 1983-06-17 and 2024-10-18.
Ticker AMZN is missing 255 trading days between 1997-05-15 and 2024-12-06.
Ticker AXP is missing 482 trading days between 1972-04-03 and 2024-10-18.
Ticker BA is missing 600 trading days between 1962-01-02 and 2024-10-18.
Ticker CAT is missing 600 trading days between 1962-01-02 and 2024-10-18.
Ticker CRM is missing 186 trading days between 2004-06-23 and 2024-10-18.
Ticker CSCO is missing 312 trading days between 1990-02-16 and 2024-10-18.
Ticker CVX is missing 600 trading days between 1962-01-02 and 2024-12-06.
Ticker DIS is missing 576 trading days between 1962-01-02 and 2024-12-06.
Ticker GS is missing 236 trading days between 1999-05-04 and 2024-10-18.
Ticker HD is missing 378 trading days between 1981-09-22 and 2024-10-18.
Ticker HON is missing 600 trading days between 1962-01-02 and 2024-10-18.
Ticker IBM is missing 600 trading day

# Outliers finding 

In [48]:



def flag_returns(df: pd.DataFrame, threshold_std: float = 3.0) -> pd.DataFrame:
    """
    Flag extreme daily returns per ticker using z-scores:
      z = (return - mean) / std
    Flags rows where |z| > threshold_std.
    """
    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")
    d = d.sort_values(["ticker", "ts"])

    # Daily returns per ticker (pct)
    d["returns"] = d.groupby("ticker")["close"].pct_change()

    # Mean/std per ticker
    g = d.groupby("ticker")["returns"]
    d["ret_mean"] = g.transform("mean")
    d["ret_std"] = g.transform("std")

    # z-score (avoid division by zero)
    d["z_score"] = np.where(d["ret_std"] > 0, (d["returns"] - d["ret_mean"]) / d["ret_std"], np.nan)

    # Flag extremes
    out = d.loc[d["z_score"].abs() > threshold_std, ["ticker", "ts", "close", "returns", "z_score"]].copy()
    out = out.rename(columns={"ts": "date"})
    out["return_pct"] = out["returns"] * 100

    return out.sort_values(["ticker", "date"]).reset_index(drop=True)


def impossible_prices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detect impossible OHLC relationships.

    For valid OHLC data (daily bar):
      high must be >= open and >= close
      low  must be <= open and <= close
    """
    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")

    checks = {
        "high < close": d["high"] < d["close"],
        "low > close": d["low"] > d["close"],
        "high < open": d["high"] < d["open"],
        "low > open": d["low"] > d["open"],
    }

    frames = []
    cols = ["ticker", "ts", "open", "high", "low", "close"]

    for issue, mask in checks.items():
        tmp = d.loc[mask, cols].copy()
        if not tmp.empty:
            tmp["issue"] = issue
            frames.append(tmp)

    if not frames:
        return pd.DataFrame(columns=["ticker", "date", "issue", "open", "high", "low", "close"])

    out = pd.concat(frames, ignore_index=True).rename(columns={"ts": "date"})
    return out.sort_values(["ticker", "date", "issue"]).reset_index(drop=True)


def negative_prices(df: pd.DataFrame, price_cols=("open", "high", "low", "close")) -> pd.DataFrame:
    """
    Flag rows where any price column is negative.
    Output is in 'long' format: one row per (ticker, date, column) violation.
    """
    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")

    # Build a long table of prices
    long_prices = d[["ticker", "ts", *price_cols]].melt(
        id_vars=["ticker", "ts"],
        value_vars=list(price_cols),
        var_name="field",
        value_name="value",
    )

    out = long_prices.loc[long_prices["value"] < 0].copy()
    out = out.rename(columns={"ts": "date"})
    out["issue"] = "negative_" + out["field"]

    return out[["ticker", "date", "issue", "value"]].sort_values(["ticker", "date", "issue"]).reset_index(drop=True)


# ----------------------------
# Run checks (clean notebook output)
# ----------------------------
potential_outliers = flag_returns(df, threshold_std=3)
print(f"Potential outliers: {len(potential_outliers)}")
display(potential_outliers.head(10))

impossible_ohlc = impossible_prices(df)
print(f"\nImpossible prices: {len(impossible_ohlc)}")
display(impossible_ohlc.head(10))

negatives = negative_prices(df)
print(f"\nNegative prices: {len(negatives)}")
display(negatives.head(10))


Potential outliers: 5741


,ticker,date,close,returns,z_score,return_pct
0,AAPL,1980-12-26,0.158480,0.092288,3.408099,9.228755
1,AAPL,1981-04-02,0.117750,0.087659,3.235323,8.765934
2,AAPL,1981-09-25,0.063616,-0.129776,-4.881801,-12.977580
3,AAPL,1981-10-02,0.073661,0.081977,3.023197,8.197709
4,AAPL,1981-12-17,0.094308,0.083340,3.074078,8.334003
5,AAPL,1981-12-18,0.102120,0.082835,3.055223,8.283497
6,AAPL,1982-03-19,0.074219,0.090173,3.329173,9.017333
7,AAPL,1982-11-22,0.125560,-0.089089,-3.362904,-8.908880
8,AAPL,1982-11-30,0.142300,0.103871,3.840521,10.387092
9,AAPL,1983-01-20,0.166850,0.111518,4.126004,11.151822



Impossible prices: 5


,ticker,date,open,high,low,close,issue
0,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,low > open
1,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,low > open
2,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,low > open
3,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,low > open
4,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,low > open



Negative prices: 0


,ticker,date,issue,value


## Understanding Inconsistencies

In [49]:
def inconsistencies(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag rows where open/close are outside the [low, high] range.
    Returns a tidy table with one row per issue.
    """
    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")
    d = d.sort_values(["ticker", "ts"])

    # Masks for inconsistencies (ignore rows where any needed value is missing)
    req = d[["low", "high", "open", "close"]].notna().all(axis=1)

    open_bad = req & ~d["open"].between(d["low"], d["high"], inclusive="both")
    close_bad = req & ~d["close"].between(d["low"], d["high"], inclusive="both")

    cols = ["ticker", "ts", "low", "open", "close", "high"]

    out = []

    if open_bad.any():
        tmp = d.loc[open_bad, cols].copy()
        tmp["issue"] = "open not in [low, high]"
        out.append(tmp)

    if close_bad.any():
        tmp = d.loc[close_bad, cols].copy()
        tmp["issue"] = "close not in [low, high]"
        out.append(tmp)

    if not out:
        return pd.DataFrame(columns=["ticker", "date", "issue", "low", "open", "close", "high"])

    result = pd.concat(out, ignore_index=True).rename(columns={"ts": "date"})
    return result.sort_values(["ticker", "date", "issue"]).reset_index(drop=True)


# Run check (clean notebook output)
incons_df = inconsistencies(df)
print(f"Inconsistencies found: {len(incons_df)}")
display(incons_df.head(20))


Inconsistencies found: 5


,ticker,date,low,open,close,high,issue
0,CVX,1962-01-02,3.24000,0.0,3.30000,3.30000,"open not in [low, high]"
1,JNJ,1962-01-02,0.22222,0.0,0.22338,0.22338,"open not in [low, high]"
2,MCD,1966-07-05,0.26749,0.0,0.26955,0.27366,"open not in [low, high]"
3,MRK,1962-01-02,0.39317,0.0,0.39537,0.39703,"open not in [low, high]"
4,PG,1962-01-02,1.42000,0.0,1.43000,1.44000,"open not in [low, high]"


# Data Report

In [50]:
def create_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a comprehensive data quality report by combining:
      - extreme returns
      - impossible prices
      - price inconsistencies
      - negative prices
      - zero/missing volume

    Output columns:
      ticker, issue_type, issue_date, description, severity
    """
    issues = []

    # ----------------------------
    # Extreme returns (medium)
    # ----------------------------
    extreme_return = flag_returns(df)
    if not extreme_return.empty:
        er = extreme_return.copy()
        er["issue_type"] = "extreme_return"
        er["severity"] = "medium"
        er["issue_date"] = er["date"]
        er["description"] = er.apply(
            lambda r: f"Return: {r['return_pct']:.2f}% (z-score:{r['z_score']:.2f})",
            axis=1,
        )
        issues.append(er[["ticker", "issue_type", "issue_date", "description", "severity"]])

    # ----------------------------
    # Impossible OHLC prices (high)
    # ----------------------------
    imp = impossible_prices(df)
    if not imp.empty:
        ip = imp.copy()
        ip["issue_type"] = "impossible_price"
        ip["severity"] = "high"
        ip["issue_date"] = ip["date"]
        ip["description"] = ip["issue"]
        issues.append(ip[["ticker", "issue_type", "issue_date", "description", "severity"]])

    # ----------------------------
    # Price inconsistencies (medium)
    # ----------------------------
    inc = inconsistencies(df)
    if not inc.empty:
        pi = inc.copy()
        pi["issue_type"] = "price_inconsistency"
        pi["severity"] = "medium"
        pi["issue_date"] = pi["date"]
        pi["description"] = pi["issue"]
        issues.append(pi[["ticker", "issue_type", "issue_date", "description", "severity"]])

    # ----------------------------
    # Negative prices (high)
    # FIX: use negative_prices(), not non_negative()
    # ----------------------------
    neg = negative_prices(df)
    if not neg.empty:
        np_ = neg.copy()
        np_["issue_type"] = "negative_price"
        np_["severity"] = "high"
        np_["issue_date"] = np_["date"]
        np_["description"] = np_["issue"]
        issues.append(np_[["ticker", "issue_type", "issue_date", "description", "severity"]])

    # ----------------------------
    # Zero or missing volume (low)
    # ----------------------------
    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")

    zero_vol_mask = d["volume"].isna() | (d["volume"] == 0)
    if zero_vol_mask.any():
        zv = d.loc[zero_vol_mask, ["ticker", "ts"]].copy()
        zv["issue_type"] = "zero_volume"
        zv["severity"] = "low"
        zv["issue_date"] = zv["ts"]
        zv["description"] = "Zero or missing volume"
        issues.append(zv[["ticker", "issue_type", "issue_date", "description", "severity"]])

    # ----------------------------
    # Combine into final report
    # ----------------------------
    if issues:
        quality_report = pd.concat(issues, ignore_index=True)
        quality_report["issue_date"] = pd.to_datetime(
            quality_report["issue_date"], errors="coerce"
        )
        quality_report = quality_report.sort_values(
            ["severity", "ticker", "issue_date", "issue_type"],
            ascending=[True, True, True, True],
        ).reset_index(drop=True)
    else:
        quality_report = pd.DataFrame(
            columns=["ticker", "issue_type", "issue_date", "description", "severity"]
        )

    # ----------------------------
    # Summary output
    # ----------------------------
    print("\n=== DATA QUALITY REPORT ===")
    print(f"Total issues: {len(quality_report)}")

    if not quality_report.empty:
        print("\nBy severity:")
        display(quality_report["severity"].value_counts())

        print("\nBy type:")
        display(quality_report["issue_type"].value_counts())
    else:
        print("No issues detected.")

    return quality_report


# Run the report
quality_report = create_quality_report(df)

print("\nSample issues (first 10):")
display(quality_report.head(10))



=== DATA QUALITY REPORT ===
Total issues: 5832

By severity:


severity
medium    5746
low         81
high         5
Name: count, dtype: int64


By type:


issue_type
extreme_return         5741
zero_volume              81
impossible_price          5
price_inconsistency       5
Name: count, dtype: int64


Sample issues (first 10):


,ticker,issue_type,issue_date,description,severity
0,CVX,impossible_price,1962-01-02,low > open,high
1,JNJ,impossible_price,1962-01-02,low > open,high
2,MCD,impossible_price,1966-07-05,low > open,high
3,MRK,impossible_price,1962-01-02,low > open,high
4,PG,impossible_price,1962-01-02,low > open,high
5,AXP,zero_volume,1972-12-14,Zero or missing volume,low
6,AXP,zero_volume,1972-12-14,Zero or missing volume,low
7,HON,zero_volume,1982-09-23,Zero or missing volume,low
8,HON,zero_volume,1982-09-24,Zero or missing volume,low
9,IBM,zero_volume,1973-10-09,Zero or missing volume,low


# Data Cleaning

In [51]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Handle missing values in OHLCV data:
      1) Drop rows where ALL core OHLC prices are missing
      2) Forward-fill prices within each ticker (time-ordered)
      3) Fill missing volume with 0
    Returns a cleaned copy of the input DataFrame.
    """
    price_cols = ["open", "high", "low", "close"]
    ffill_cols = ["open", "high", "low", "close", "adj_close"]

    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")
    d = d.sort_values(["ticker", "ts"]).reset_index(drop=True)

    # 1) Drop rows where all OHLC are missing
    rows_before = len(d)
    drop_mask = d[price_cols].isna().all(axis=1)
    dropped = int(drop_mask.sum())
    d = d.loc[~drop_mask].copy()
    print(f"Dropped rows where all OHLC are missing: {dropped:,} (out of {rows_before:,})")

    # 2) Forward fill prices within each ticker (vectorized)
    before_missing = d[ffill_cols].isna().sum()
    d[ffill_cols] = d.groupby("ticker", sort=False)[ffill_cols].ffill()
    after_missing = d[ffill_cols].isna().sum()

    # Report only columns that changed / had missing values
    changes = pd.DataFrame({"before": before_missing, "after": after_missing})
    changes = changes.loc[changes["before"] > 0]
    if not changes.empty:
        print("\nMissing values after forward-fill (by column):")
        display(changes)

    # 3) Fill volume with 0
    vol_before = int(d["volume"].isna().sum())
    d["volume"] = d["volume"].fillna(0)
    print(f"\nVolume missing filled with 0: {vol_before:,} → 0")

    return d


df_cleaned = handle_missing_values(df)


Dropped rows where all OHLC are missing: 0 (out of 397,711)

Missing values after forward-fill (by column):


,before,after
adj_close,19855,1



Volume missing filled with 0: 0 → 0


# Approach to Handle Outliers

In [52]:
def handle_outliers(df: pd.DataFrame, *, fix_cols=("open", "high", "low"), verbose: bool = True) -> pd.DataFrame:
    """
    Handle outliers / impossible OHLC relationships in a more pythonic way.

    Strategy:
      1) Identify impossible OHLC conditions and set the offending field(s) to NaN
      2) Fill created NaNs within each ticker using ffill then bfill (to cover first rows)
      3) Flag extreme returns (kept; reported for review)
    """
    d = df.copy()
    d["ts"] = pd.to_datetime(d["ts"], errors="coerce")
    d = d.sort_values(["ticker", "ts"]).reset_index(drop=True)

    # ---- Step 1: Replace impossible values with NaN (vectorized) ----
    # Each rule is: (name, boolean_mask, column_to_null)
    rules = [
        ("open=0 causes low>open", (d["open"] == 0) & (d["low"] > d["open"]), "open"),
        ("high < close",            d["high"] < d["close"],                   "high"),
        ("low > close",             d["low"] > d["close"],                    "low"),
        ("high < open",             d["high"] < d["open"],                    "high"),
        ("low > open",              d["low"] > d["open"],                     "low"),
    ]

    fixes_summary = []
    for name, mask, col in rules:
        n = int(mask.sum())
        if n:
            d.loc[mask, col] = np.nan
        fixes_summary.append({"rule": name, "column_set_nan": col, "count": n})

    fixes_df = pd.DataFrame(fixes_summary)
    if verbose:
        print("Impossible-price fixes applied (set to NaN):")
        display(fixes_df[fixes_df["count"] > 0] if (fixes_df["count"] > 0).any() else fixes_df.head(0))

    # ---- Step 2: Fill NaNs created by fixes (within each ticker) ----
    before_missing = d[list(fix_cols)].isna().sum()
    d[list(fix_cols)] = d.groupby("ticker", sort=False)[list(fix_cols)].ffill()
    d[list(fix_cols)] = d.groupby("ticker", sort=False)[list(fix_cols)].bfill()
    after_missing = d[list(fix_cols)].isna().sum()

    fill_report = pd.DataFrame({"before": before_missing, "after": after_missing})
    fill_report = fill_report.loc[fill_report["before"] > 0]
    if verbose and not fill_report.empty:
        print("\nMissing values in price columns after fill (ffill → bfill):")
        display(fill_report)

    # ---- Step 3: Flag extreme returns (keep, but report) ----
    extreme = flag_returns(d)

    if verbose:
        print(f"\nFound {len(extreme)} extreme returns.")
        print("Strategy: Keeping all extreme returns (may represent valid events).")
        print("In production, investigate: splits, corporate actions, major news, data errors.\n")

        if not extreme.empty and "z_score" in extreme.columns:
            top_extreme = extreme.reindex(columns=["ticker", "date", "return_pct", "z_score"], fill_value=np.nan) \
                                 .assign(abs_z=lambda x: x["z_score"].abs()) \
                                 .sort_values("abs_z", ascending=False) \
                                 .head(10) \
                                 .drop(columns=["abs_z"])
            print("Top 10 extreme returns by |z_score|:")
            display(top_extreme)

    return d


# Apply outlier handling
df_cleaned = handle_outliers(df_cleaned)


Impossible-price fixes applied (set to NaN):


,rule,column_set_nan,count
0,open=0 causes low>open,open,5
4,low > open,low,5



Missing values in price columns after fill (ffill → bfill):


,before,after
open,5,0
low,5,0



Found 5741 extreme returns.
Strategy: Keeping all extreme returns (may represent valid events).
In production, investigate: splits, corporate actions, major news, data errors.

Top 10 extreme returns by |z_score|:


,ticker,date,return_pct,z_score
5034,TRV,1981-08-10,100.000000,42.887840
5029,TRV,1976-10-28,100.000000,42.887840
5032,TRV,1978-12-20,99.751244,42.781097
3485,MCD,1968-01-02,-75.362033,-42.290343
4689,PG,2000-03-07,-31.366674,-24.101394
5035,TRV,1981-08-11,-50.000000,-21.478056
4621,PG,1987-10-19,-27.703985,-21.290358
5030,TRV,1976-10-29,-49.132321,-21.105730
5033,TRV,1978-12-21,-49.066002,-21.077272
108,AAPL,2000-09-29,-51.869452,-19.400619


# Validation

In [53]:
def validate_and_save_cleaned_data(df_raw: pd.DataFrame, df_cleaned: pd.DataFrame, conn) -> None:
    """
    Short validation + save:
      - missing OHLC
      - negative OHLC
      - OHLC relationship checks (high/low vs open/close)
      - before/after summary
      - save to ohlc_hw_cleaned
    """
    table_name = "ohlc_hw_cleaned"
    price_cols = ["open", "high", "low", "close"]

    raw = df_raw.copy()
    clean = df_cleaned.copy()

    # ---- Validations (vectorized) ----
    missing_prices = int(clean[price_cols].isna().sum().sum())
    negative_prices_cnt = int((clean[price_cols] < 0).sum().sum())

    rel_issues = {
        "high < close": int((clean["high"] < clean["close"]).sum()),
        "low > close": int((clean["low"] > clean["close"]).sum()),
        "high < open": int((clean["high"] < clean["open"]).sum()),
        "low > open": int((clean["low"] > clean["open"]).sum()),
    }
    total_rel_issues = sum(rel_issues.values())

    # ---- Before/after stats ----
    rows_before, rows_after = len(raw), len(clean)
    missing_before = int(raw[price_cols].isna().sum().sum())
    missing_after = int(clean[price_cols].isna().sum().sum())

    # NOTE: use your updated function names (impossible_prices / inconsistencies)
    issues_before = len(impossible_prices(raw)) + len(inconsistencies(raw))
    issues_after = len(impossible_prices(clean)) + len(inconsistencies(clean))

    # ---- Print summary ----
    print("\n=== VALIDATION SUMMARY ===")
    print(f"Missing OHLC: {missing_prices} {'✓' if missing_prices==0 else '✗'}")
    print(f"Negative OHLC: {negative_prices_cnt} {'✓' if negative_prices_cnt==0 else '✗'}")
    print(f"OHLC relationship issues: {total_rel_issues} {'✓' if total_rel_issues==0 else '✗'}")
    if total_rel_issues:
        for k, v in rel_issues.items():
            print(f"  - {k}: {v}")

    print("\n=== BEFORE / AFTER ===")
    print(f"Rows: {rows_before:,} → {rows_after:,} (dropped {rows_before-rows_after:,})")
    print(f"Missing OHLC: {missing_before:,} → {missing_after:,}")
    print(f"Impossible/Inconsistent: {issues_before:,} → {issues_after:,} (fixed {issues_before-issues_after:,})")

    # ---- Save (replace table) ----
    clean.to_sql(table_name, conn, index=False, if_exists="replace")
    saved_rows = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {table_name}", conn)["n"].iloc[0]
    print(f"\n✓ Saved to '{table_name}' ({saved_rows:,} rows)")


# Run validation and save
validate_and_save_cleaned_data(df, df_cleaned, conn)



=== VALIDATION SUMMARY ===
Missing OHLC: 0 ✓
Negative OHLC: 0 ✓
OHLC relationship issues: 0 ✓

=== BEFORE / AFTER ===
Rows: 397,711 → 397,711 (dropped 0)
Missing OHLC: 0 → 0
Impossible/Inconsistent: 10 → 0 (fixed 10)

✓ Saved to 'ohlc_hw_cleaned' (397,711 rows)


In [54]:
# ----------------------------
# Connect to database
# ----------------------------
DB_PATH = "data3.db"
TABLE = "ohlc_hw_cleaned"

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql(
        f"""
        SELECT
            ts,
            open,
            high,
            low,
            close,
            adj_close,
            volume,
            ticker
        FROM {TABLE}
        ORDER BY ts
        """,
        conn
    )

# ----------------------------
# Basic type handling
# ----------------------------
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

# ----------------------------
# Quick inspection
# ----------------------------
display(df.head())
df.info()
df.describe(include="all")


,ts,open,high,low,close,adj_close,volume,ticker
0,1962-01-02,0.837449,0.837449,0.823045,0.823045,0.190931,352350,BA
1,1962-01-02,1.600000,1.620000,1.590000,1.600000,0.476660,163200,CAT
2,1962-01-02,3.300000,3.300000,3.270000,3.300000,0.335750,105840,CVX
3,1962-01-02,0.092908,0.096026,0.092908,0.092908,0.058209,841958,DIS
4,1962-01-02,8.330000,8.330000,8.270000,8.310000,2.080000,40740,HON


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  397710 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,397710.000000,3.977110e+05,397711
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DIS
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17151
mean,1998-04-17 21:18:45.292486144,46.285493,46.747660,45.816279,46.292903,51.583098,3.097773e+07,NaN
min,1962-01-02 00:00:00,0.005208,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.970000,4.648200e+06,NaN
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.780000,1.102380e+07,NaN
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09,NaN


In [55]:
# get copy of new dataset to use in next python codes 

In [56]:
import sqlite3

# connect to database
conn = sqlite3.connect("data3.db")

# create cursor
cursor = conn.cursor()

# get table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

print(tables)

[('ohlc',), ('ohlc_hw',), ('market_caps',), ('ohlc_hw_cleaned',)]


In [57]:
import shutil

source = "data3.db"
destination = "cleaned_database.db"

shutil.copy(source, destination)

print("Database copied successfully.")

Database copied successfully.


In [58]:
import sqlite3

# connect to database
conn = sqlite3.connect("cleaned_database.db")

# create cursor
cursor = conn.cursor()

# get table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

tables = cursor.fetchall()

print(tables)

[('ohlc',), ('ohlc_hw',), ('market_caps',), ('ohlc_hw_cleaned',)]


In [59]:
import sqlite3

# original database
source = sqlite3.connect("data3.db")

# new database
destination = sqlite3.connect("data3_clean.db")

# copy everything
source.backup(destination)

destination.close()
source.close()

print("Database copied successfully")

Database copied successfully


In [60]:
import shutil

source = "data3.db"
destination = "data3_cleaned.db"

shutil.copy(source, destination)

print("Database copied successfully.")

Database copied successfully.


In [61]:
import sqlite3
conn = sqlite3.connect('data3_clean.db')